# Models Experimentation Notebook for IFood Data Science Technical Case

## 1. Import Libraries

In [1]:
import numpy as np
from itertools import chain
from collections import Counter
import matplotlib.pyplot as plt

from pyspark.sql.functions import col, sum, when, count, to_date, year, avg, from_json, coalesce, corr,create_map, lit, dense_rank
from pyspark.sql import Window as W
from pyspark.sql import SparkSession
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.recommendation import ALS
from pyspark.ml.tuning import TrainValidationSplit, ParamGridBuilder

# Cria a sessão Spark local
spark = SparkSession.builder.appName("ifoodProcessingData").master("local[*]").getOrCreate()

# Verifica
print(spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/09/27 17:29:47 WARN Utils: Your hostname, marianna-pinho, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
25/09/27 17:29:47 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/09/27 17:29:55 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/09/27 17:29:58 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
25/09/27 17:29:58 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


4.0.1


## 2. Utilities

In [2]:
def get_columns_with_nulls(dataframe):
    return dataframe.select([sum(col(c).isNull().cast("int")).alias(c) for c in dataframe.columns])

def info_alternative(dataframe):
    n_rows = dataframe.count()
    print(f"Number of rows: {n_rows}")
    print(f"Number of columns: {len(dataframe.columns)}")
    number_of_nans_in_cols = get_columns_with_nulls(dataframe)
    print("#\tColumn\t\tNon-Null Count\t\tDtype")
    print("---\t------\t\t--------------\t\t-----")
    for index, (col, col_type) in enumerate(dataframe.dtypes):
        print(f"{index}\t{col}\t\t{n_rows - number_of_nans_in_cols.first()[col]}\t\t{col_type}")


def create_id_int_column(data, column_map, original_column_name, new_column_name):
    mapping_expr = create_map([lit(x) for x in chain(*column_map.items())])

    return data.withColumn(
        new_column_name,
        coalesce(
            mapping_expr[col(original_column_name)],
            lit(-1)   # default se não achar a chave
            )
        )

In [3]:
def plot_bars(axes, data_values, data_labels, data_colors, title, x_label, y_label):
    axes.bar(x=data_labels, height=data_values, color=data_colors)
    axes.set_title(title)
    axes.set_xlabel(x_label)
    axes.set_ylabel(y_label)

    return axes

def plot_hist(axes, data_values, n_bins, title, x_label, y_label, data_mean=None, data_median=None, mean_color="tab:red", median_color="tab:green"):

    axes.hist(data_values, bins=n_bins)

    if data_mean is not None:
        axes.axvline(x=data_mean, color=mean_color, linestyle="dashed")
        axes.text(data_mean, axes.get_ylim()[1]-100, "Média", color=mean_color)
    if data_median is not None:
        axes.axvline(x=data_median, color=median_color, linestyle="dashed")
        axes.text(data_median, axes.get_ylim()[1]-200, "Mediana", color=median_color)

    axes.set_title(title)
    axes.set_xlabel(x_label)
    axes.set_ylabel(y_label)

    return axes

## 3. Loading Data

## 4. Training Models

In [ ]:
df_transactions_offers = df_transactions.filter(col("event") != "transaction")
info_alternative(df_transactions_offers)

type_event_map = {"offer received": 1, "offer viewed": 3, "offer completed": 5}
df_transactions_offers = create_id_int_column(data=df_transactions_offers, column_map=type_event_map, original_column_name="event", new_column_name="event_int")

fact_mat_client_offer  = (
    df_transactions_offers
    .groupBy("account_id", "offer_id")
    .agg(avg("event_int").alias("event_levels"))
)

In [ ]:
unique_account_ids =  fact_mat_client_offer.select("account_id").distinct().rdd.flatMap(lambda x: x).collect()
account_id_map = {val: idx for idx, val in enumerate(unique_account_ids)}

unique_offer_ids =  fact_mat_client_offer.select("offer_id").distinct().rdd.flatMap(lambda x: x).collect()
offer_id_map = {val: idx for idx, val in enumerate(unique_offer_ids)}

fact_mat_client_offer = create_id_int_column(data=fact_mat_client_offer, column_map=account_id_map, original_column_name="account_id", new_column_name="account_id_int")
fact_mat_client_offer = create_id_int_column(data=fact_mat_client_offer, column_map=offer_id_map, original_column_name="offer_id", new_column_name="offer_id_int")

In [ ]:
(training, test) = fact_mat_client_offer.randomSplit([0.8, 0.2])
training.count(), test.count()

In [ ]:
als_model = ALS(
    userCol="account_id_int",
    itemCol="offer_id_int",
    ratingCol="event_levels",
    coldStartStrategy="drop",
    nonnegative=True
)

In [ ]:
param_grid = ParamGridBuilder()\
    .addGrid(als_model.rank, [12,14])\
    .addGrid(als_model.maxIter, [18,20])\
    .addGrid(als_model.regParam, [.17,.19])\
    .build()

In [ ]:
## Não quero predizer os levels, quero predizer a oferta. 
evaluator = RegressionEvaluator(metricName="rmse", labelCol="event_levels", predictionCol="prediction")

In [ ]:
tvs = TrainValidationSplit(estimator=als_model, estimatorParamMaps=param_grid, evaluator=evaluator)

In [ ]:
training.cache()

In [ ]:
model = tvs.fit(training)

In [ ]:
best_model = model.bestModel
best_model

In [ ]:
predictions = best_model.transform(test)

In [ ]:
rmse = evaluator.evaluate(predictions)
rmse

In [ ]:
user_recs = best_model.recommendForAllUsers(3)

In [ ]:
list(filter(lambda key: account_id_map[key] == 1, account_id_map))


In [ ]:
list(filter(lambda key: offer_id_map[key] == 4, offer_id_map))

## 5. Evaluating Models